In [ ]:
# import libraries
import os
import pandas as pd
import numpy as np
import seaborn as sns
sns.set(color_codes=True)
import matplotlib.pyplot as plt
%matplotlib inline
import pickle
import tensorflow as tf

file_name = "run57_mix_mega_shared"
model_name = "run53_mix_mega_shared_tanh.keras"
normalize= 1
test_number = 100000
os.environ["CUDA_VISIBLE_DEVICES"]="-1"
number_of_detectors = 6
data_dir = "/data/test_newrepo"

In [ ]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
file_path = data_dir+'/'+file_name+'_dataset.pkl'

# Load the dataset from the pickle file
with open(file_path, 'rb') as file:
    loaded_array = pickle.load(file)

In [ ]:
# Filter the dataset by spectral model and flux

loaded_array_test = loaded_array
filter_flux = 0 #1:30
filter_spectra = 0

filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

count = 0

if filter_flux == 1:
    for grb in loaded_array_test:
        if grb['flux'] > 20 and grb['flux'] <= 30:
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

count = 0

if filter_spectra==1:
    for grb in loaded_array_test:
        if  "1500" in grb['spectrum']: #["Band 10 10000 -1.9 -3.7 230","Band 10 10000 -1 -2.3 699.9","Comptonized 10 10000 -0.5 1500"]
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

In [ ]:
loaded_array_test.shape

In [ ]:
# extract a random sample from the testing dataset
filter_len = test_number
random_indices = np.random.permutation(len(loaded_array_test))
shuffled_loaded_array_bkg = loaded_array_test[random_indices]
test_dataset_raw = shuffled_loaded_array_bkg[:filter_len]              

In [ ]:
test_dataset_raw.shape

In [ ]:
def diff_phi(a1, a2):
    diff = abs(a1 - a2)
    if diff > 180:
        diff = 360 - diff
    return diff

def convert_degrees_to_sin_cos(degrees):
    radians = np.deg2rad(degrees)  
    sin = np.sin(radians)
    cos = np.cos(radians)
    return sin, cos

def degrees_from_sin_cos(sin_value, cos_value):
  
    angle_rad = np.arctan2(sin_value, cos_value)
    
    angle_deg = np.degrees(angle_rad)
   
    if angle_deg < 0:
        angle_deg += 360
    return angle_deg

def convert_degress_to_sin_cos_math(degrees):
    
    rad = math.radians(degrees)

    
    sin = math.sin(rad)
    cos = math.cos(rad)

    print("Seno:", sin)
    print("Coseno:", cos)


def angular_distance(theta1, phi1, theta2, phi2):
    """
    Compute the angular (great-circle) distance in degrees between two points
    specified by (theta, phi) coordinates in degrees.

    Parameters:
        theta1, phi1: floats, coordinates of the first point (degrees).
        theta2, phi2: floats, coordinates of the second point (degrees).

    Returns:
        angular_dist_deg: float, angular distance between the two points in degrees.
    """
    # Adjust theta values by shifting -90
    theta1 = 90 - theta1
    theta2 = 90 - theta2

    # Convert angles to radians
    theta1_rad = math.radians(theta1)
    phi1_rad = math.radians(phi1)
    theta2_rad = math.radians(theta2)
    phi2_rad = math.radians(phi2)
    
    # Compute the difference in longitude
    delta_phi = abs(phi1_rad - phi2_rad)

    # Apply spherical law of cosines
    value = (math.sin(theta1_rad) * math.sin(theta2_rad) +
             math.cos(theta1_rad) * math.cos(theta2_rad) * math.cos(delta_phi))
    
    # Clamp value to [-1, 1] to avoid floating point errors in acos
    value_clamped = max(-1.0, min(1.0, value)) 
    
    angular_dist = math.acos(value_clamped)
    
    # Convert angular distance from radians to degrees
    angular_dist_deg = math.degrees(angular_dist)
    
    return angular_dist_deg
    

def l2_normalize(data):
  """
  Normalize a NumPy array using the L2 (Euclidean) norm.

  Args:
    data (numpy.ndarray): The array to normalize. Can be a 1D vector
                         or a 2D matrix (where each row is a vector to normalize).

  Returns:
    numpy.ndarray: The L2-normalized array.
  """

  data = np.array(data)
  if data.ndim == 1:
    # Case: 1D vector
    norm = np.sqrt(np.sum(data**2))
    if norm == 0:
      return data  # Avoid division by zero if the vector is null
    return data / norm
  elif data.ndim == 2:
    # Case: 2D matrix (normalize each row)
    norms = np.sqrt(np.sum(data**2, axis=1, keepdims=True))
    # Handle the case of null norms (zero rows)
    norms[norms == 0] = 1
    return data / norms
  else:
    raise ValueError("Input must be a 1D or 2D array.")

    
def prepare_dataset(data_array):

    dataset = np.empty((len(data_array),number_of_detectors))
    labels = np.empty((len(data_array),2))
    
    count = -1
    for element in data_array:
        
        count+=1
        # We are considering a Poissonian background. The mean counts are calculated from the DC3 BGO data.
        b_sim = np.array([57.6053, 58.7157, 51.4131, 48.2891, 47.7293, 45.9617])
        dataset[count] = l2_normalize(np.array(element['counts'])+np.random.poisson(b_sim*20)-b_sim*20)
        #dataset[count] = l2_normalize(element['counts'])
        labels[count][0] = float(element['coord'][0])
        labels[count][1] = float(element['coord'][1])
        
    labels = labels[:count+1]
    dataset = dataset[:count+1]

    # Convert to radians
    coords_rad = []
    
    for theta, phi in labels:
        coords_rad.append([theta, phi])
    
    theta, phi = zip(*coords_rad)
  
    labels_cosin = np.empty((len(labels),4))
    count = -1
    for element in labels:
        
        count=count+1
        
        theta_sin,theta_cos = convert_degrees_to_sin_cos(labels[count][0])
        labels_cosin[count][0] = theta_sin
        labels_cosin[count][1] = theta_cos
        
        phi_sin,phi_cos = convert_degrees_to_sin_cos(labels[count][1])
        labels_cosin[count][2] = phi_sin
        labels_cosin[count][3] = phi_cos
    
    labels_norm = labels_cosin
    
    if normalize == 1:
    
        labels_norm = np.empty((len(labels_cosin),4))
    
        for j in range(0,len(labels_cosin)):
            labels_norm[j][0] = 2 * labels_cosin[j][0] - 1
            labels_norm[j][1] = labels_cosin[j][1]
            labels_norm[j][2] = labels_cosin[j][2]
            labels_norm[j][3] = labels_cosin[j][3]
        
    else:
        labels_norm = labels_cosin

    
   
    return dataset, labels, labels_norm, theta, phi, coords_rad


In [ ]:
test_dataset, labels, test_labels, theta, phi, coords_rad = prepare_dataset(test_dataset_raw)

In [ ]:
test_dataset.shape

In [ ]:
plt.hist(theta)
plt.xlabel("Theta")
plt.ylabel("Counts")

In [ ]:
plt.hist(phi)
plt.xlabel("Phi")
plt.ylabel("Counts")

In [ ]:
#scale labels and convert in cos and sin
max_lon = 180.0
min_lon = 0.0

max_lat = 360.0
min_lat = 0.0

labels_cosin = np.empty((len(labels),4))
count = -1
for element in labels:
    
    count=count+1
    
    theta_sin,theta_cos = convert_degrees_to_sin_cos(labels[count][0])
    labels_cosin[count][0] = theta_sin
    labels_cosin[count][1] = theta_cos
    
    phi_sin,phi_cos = convert_degrees_to_sin_cos(labels[count][1])
    labels_cosin[count][2] = phi_sin
    labels_cosin[count][3] = phi_cos

labels_norm = labels_cosin

In [ ]:
if normalize == 1:

    labels_norm = np.empty((len(labels),4))

    min_max_values = []


    for j in range(0,len(labels_cosin)):
        labels_norm[j][0] = 2 * labels_cosin[j][0] - 1
        labels_norm[j][1] = labels_cosin[j][1]
        labels_norm[j][2] = labels_cosin[j][2]
        labels_norm[j][3] = labels_cosin[j][3]
    
else:
    labels_norm = labels_cosin


In [ ]:
plt.hist(labels_norm[:,0],bins=100)

In [ ]:
plt.hist(labels_norm[:,1],bins=100)

In [ ]:
plt.hist(labels_norm[:,2],bins=100)

In [ ]:
plt.hist(labels_norm[:,3],bins=100)

In [ ]:
from tensorflow.keras import backend as K

def spherical_loss_2angles(y_true, y_pred):
    # Compute the Mean Squared Error (MSE) between targets and predictions
    mse_loss = K.mean(K.square(y_true - y_pred), axis=-1)
    
    # a the pairs (sinθ, cosθ) and (sinφ, cosφ) from the predictions
    sin_theta, cos_theta = y_pred[:, 0], y_pred[:, 1]
    sin_phi, cos_phi     = y_pred[:, 2], y_pred[:, 3]
    
    # Compute the L2 norm of each pair
    # (should ideally be equal to 1 if the network outputs valid sine/cosine pairs)
    norm_theta = K.sqrt(sin_theta**2 + cos_theta**2)
    norm_phi   = K.sqrt(sin_phi**2 + cos_phi**2)
    
    # Compute the penalty as the squared deviation of each norm from 1
    # This encourages the network to output normalized sine/cosine pairs
    penalty_theta = K.square(norm_theta - 1.0)
    penalty_phi   = K.square(norm_phi - 1.0)
    
    penalty = penalty_theta + penalty_phi
    
    # Weight of the regularization term (can be tuned as a hyperparameter)
    alpha = 0.01
    
    # Final loss = MSE + weighted normalization penalty
    return mse_loss + alpha * penalty

In [ ]:
from tensorflow.keras.models import load_model

# Load the model back from the saved directory
model = load_model(data_dir+"/"+model_name,custom_objects={'spherical_loss_2angles': spherical_loss_2angles})

In [ ]:
pred_data = model.predict(test_dataset)

In [ ]:
test_dataset.shape

In [ ]:
import math

def convert_original(pred_tmp,test_tmp):

    pred_data_renorm = np.empty((len(pred_tmp),4))
    test_data_renorm = np.empty((len(test_tmp),4))
    
        
    for j in range(0,len(pred_tmp)):
        pred_data_renorm[j][0]  = (pred_tmp[j][0] + 1) / 2
        pred_data_renorm[j][1] = pred_tmp[j][1]
        pred_data_renorm[j][2] = pred_tmp[j][2]
        pred_data_renorm[j][3] = pred_tmp[j][3]
        
    for j in range(0,len(test_tmp)):
        test_data_renorm[j][0]  = (test_tmp[j][0] + 1) / 2
        test_data_renorm[j][1] = test_tmp[j][1]
        test_data_renorm[j][2] = test_tmp[j][2]
        test_data_renorm[j][3] = test_tmp[j][3]
          
    pred_data_original_tmp = np.empty((len(pred_tmp),2))
    test_labels_original_tmp = np.empty((len(test_tmp),2))
            
    for i, element in enumerate(test_data_renorm):
        test_labels_original_tmp[i][0]=degrees_from_sin_cos(element[0],element[1])
        test_labels_original_tmp[i][1]=degrees_from_sin_cos(element[2],element[3])
    
    for i, element in enumerate(pred_data_renorm):
        
        pred_data_original_tmp[i][0]=degrees_from_sin_cos(element[0],element[1])
        pred_data_original_tmp[i][1]=degrees_from_sin_cos(element[2],element[3])
    
    absolute_diffs_theta = np.abs(pred_data_original_tmp[:, 0] - test_labels_original_tmp[:, 0])
    absolute_diffs_phi = np.abs(pred_data_original_tmp[:, 1] - test_labels_original_tmp[:, 1])
    
    # Calculate the Mean Absolute Error (MAE) for each element separately
    mae_theta = np.mean(absolute_diffs_theta)
    mae_phi = np.mean(absolute_diffs_phi)

    print("Mean Absolute Error for Theta:", mae_theta)
    print("Mean Absolute Error for Phi:", mae_phi)

    return pred_data_original_tmp , test_labels_original_tmp 

In [ ]:
pred_data_original, test_labels_original  = convert_original(pred_data,test_labels)

In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels_original[:, 0],bins=50,alpha=0.6,color="b",label="reco coords")
plt.hist(pred_data_original[:,0],bins=50,alpha=0.6,color="r",label="test coords")
plt.xlabel("Theta")
plt.legend()
plt.ylabel("Counts")


In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels_original[:, 1],bins=50,alpha=0.6,color="b",label="reco coords")
plt.hist(pred_data_original[:,1],bins=50,alpha=0.6,color="r",label="test coords")
plt.xlabel("Phi")
plt.legend()
plt.ylabel("Counts")

In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels[:,0],bins=50,alpha=0.6,color="b",label="test coords")
plt.hist(pred_data[:, 0],bins=50,alpha=0.6,color="r",label="reco coords")
plt.xlabel("Theta sin")
plt.legend()
plt.ylabel("Counts")


In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels[:,1],bins=50,alpha=0.6,color="b",label="test coords")
plt.hist(pred_data[:, 1],bins=50,alpha=0.6,color="r",label="reco coords")
plt.xlabel("Theta cos")
plt.legend()
plt.ylabel("Counts")


In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels[:,2],bins=50,alpha=0.6,color="b",label="test coords")
plt.hist(pred_data[:, 2],bins=50,alpha=0.6,color="r",label="reco coords")
plt.xlabel("Phi sin")
plt.legend()
plt.ylabel("Counts")


In [ ]:
fig = plt.figure(figsize=(8, 6))
plt.hist(test_labels[:,3],bins=50,alpha=0.6,color="b",label="test coords")
plt.hist(pred_data[:, 3],bins=50,alpha=0.6,color="r",label="reco coords")
plt.xlabel("Phi cos")
plt.legend()
plt.ylabel("Counts")


In [ ]:
# Calculate distances between corresponding coordinates

def angular_distance_phi(phi1, phi2):
    
    diff = abs(phi1 - phi2) % 360
    return min(diff, 360 - diff)

def calculated_distances(pred,test):
    distances = []
    theta_dist_arr = []
    phi_dist_arr = []

    
    for i in range(0,len(pred)):
    
        d = angular_distance(pred[i][0],pred[i][1],test[i][0],test[i][1])
        distances.append(d)
        theta_dist = np.abs(pred[i][0]-test[i][0])
        phi_dist = angular_distance_phi(pred[i][1],test[i][1])
        theta_dist_arr.append(theta_dist)
        phi_dist_arr.append(phi_dist)
        
        
    return  distances,theta_dist_arr,phi_dist_arr

In [ ]:
distances,theta_dist_arr,phi_dist_arr = calculated_distances(pred_data_original,test_labels_original)

In [ ]:
print(np.mean(distances))
print(np.mean(theta_dist_arr))
print(np.mean(phi_dist_arr))

In [ ]:
plt.hist(distances)
plt.ylabel("Counts")
plt.xlabel("Error Radius °")

In [ ]:
# Define 10-degree intervals
intervals = np.arange(0, 365, 10)

# Group counts based on 10-degree intervals
grouped_counts = np.zeros((len(intervals)))
counts = np.zeros((len(intervals)))

total_count = 0
for theta, phi, count in zip(test_labels_original[:,0], test_labels_original[:,1], distances):
    
    if (theta > 20 and theta < 55):  # Condition on theta
        total_count += 1
        interval_index = int(phi // 10)
        grouped_counts[interval_index] = grouped_counts[interval_index] + count
        counts[interval_index] += 1

# Compute the average counts in each interval
grouped_counts = grouped_counts[:total_count]
counts = counts[:total_count]

# Display histogram
plt.figure(figsize=(10, 6))
plt.bar(intervals, grouped_counts / counts, width=10, align='edge', alpha=1, label="loc. error")
plt.xlabel('Phi (°)')
plt.ylabel('Loc. error')
plt.title('Loc. error as a function of phi')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
# Define 5-degree intervals
intervals = np.arange(0, 185, 5)

# Group counts based on 5-degree intervals
grouped_counts = np.zeros((len(intervals)))
counts = np.zeros((len(intervals)))

total_count = 0
for theta, phi, count in zip(test_labels_original[:,0], test_labels_original[:,1], distances):
    
    if(True):  # (phi > 125 and phi < 145) or ...
        total_count += 1
        interval_index = int(theta // 5)
        grouped_counts[interval_index] = grouped_counts[interval_index] + count
        counts[interval_index] += 1

# Compute the average counts in each interval
grouped_counts = grouped_counts[:total_count]
counts = counts[:total_count]

# Display histogram
plt.figure(figsize=(10, 6))
plt.bar(intervals[:-1], grouped_counts[:-1] / counts[:-1], width=5, align='edge', alpha=1, label="loc. error")
plt.xlabel('Theta (°)')
plt.ylabel('Loc. error')
plt.title('Loc. error as a function of theta')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
print(np.mean(distances))
print(np.mean(theta_dist_arr))
print(np.mean(phi_dist_arr))

### 